# مشروع استخراج وتصحيح نصوص الخط اليدوي — النسخة النهائية


In [ ]:
# ============================================================
# الخلية 1: تثبيت المكتبات المطلوبة
# ============================================================
!apt-get install -y poppler-utils tesseract-ocr tesseract-ocr-ara
!pip install -q pdf2image easyocr pyspellchecker langdetect \
    transformers torch torchvision Pillow opencv-python-headless \
    pandas ar-corrector ipywidgets scikit-learn
!pip install -q peft huggingface_hub datasets


In [ ]:
# ============================================================
# الخلية 2: الاستيرادات وإعداد المسارات
# ============================================================
from google.colab import userdata
import os

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✅ تم تحميل Hugging Face Token من Colab Secrets")
except userdata.SecretNotFoundError:
    print("⚠️ لم يتم العثور على Secret باسم 'HF_TOKEN'.")
except Exception as e:
    print(f"خطأ غير متوقع: {e}")

import cv2, numpy as np, sqlite3, io, torch, json, time, logging
import pandas as pd, easyocr
from PIL import Image
from pdf2image import convert_from_path
from google.colab import drive, patches
from spellchecker import SpellChecker
from langdetect import detect, DetectorFactory
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from ar_corrector.corrector import Corrector
import ipywidgets as widgets
from IPython.display import display
from datetime import datetime

drive.mount('/content/drive')

MODEL_CACHE = '/content/drive/MyDrive/Handwriting_Dataset/models_cache'
os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ["TRANSFORMERS_CACHE"] = MODEL_CACHE
os.environ["TORCH_HOME"]         = MODEL_CACHE

PDF_PATH    = '/content/drive/MyDrive/python notes.pdf'
OUTPUT_DIR  = '/content/drive/MyDrive/Handwriting_Dataset'
LOGS_DIR    = os.path.join(OUTPUT_DIR, 'Logs')
os.makedirs(LOGS_DIR, exist_ok=True)

DB_PATH      = os.path.join(OUTPUT_DIR, 'handwriting_data.db')
LOG_FILE     = os.path.join(LOGS_DIR, f"ocr_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
FEEDBACK_CSV = os.path.join(LOGS_DIR, 'user_corrections_feedback.csv')
STATS_JSON   = os.path.join(LOGS_DIR, 'processing_stats.json')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
        logging.StreamHandler()
    ]
)

if not os.path.exists(FEEDBACK_CSV):
    pd.DataFrame(columns=[
        'timestamp', 'image_id', 'original_text', 'corrected_text', 'status'
    ]).to_csv(FEEDBACK_CSV, index=False, encoding='utf-8')

logging.info('🚀 بدء تشغيل المشروع')


In [ ]:
# ============================================================
# الخلية 3: تحميل النماذج (TrOCR، EasyOCR، المدققات)
# ============================================================
print('جاري فحص النماذج المحملة مسبقاً...')
start_time = time.time()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logging.info(f'الجهاز المستخدم: {device}')

try:
    trocr_processor = TrOCRProcessor.from_pretrained(
        'David-Magdy/TR_OCR_LARGE', cache_dir=MODEL_CACHE)
    trocr_model = VisionEncoderDecoderModel.from_pretrained(
        'David-Magdy/TR_OCR_LARGE', cache_dir=MODEL_CACHE).to(device)
    print('✅ TrOCR جاهز (من الكاش)')
except Exception as e:
    print(f'فشل تحميل TrOCR: {e}')

# تحميل LoRA إذا وُجد
lora_save_path = os.path.join(MODEL_CACHE, 'trocr_lora_finetuned')
if os.path.exists(lora_save_path):
    from peft import PeftModel
    try:
        trocr_model = PeftModel.from_pretrained(trocr_model, lora_save_path).to(device)
        print('✅ تم تحميل النموذج المُحسَّن (LoRA) من الكاش')
    except Exception as e:
        print(f'⚠️ فشل تحميل نموذج LoRA: {e}. سيتم استخدام النموذج الأساسي.')
else:
    print('ℹ️ يستخدم النموذج الأساسي (لا يوجد fine-tuning بعد)')

import os.path as osp
easy_reader   = easyocr.Reader(['en', 'ar'])
arabic_spell  = Corrector()
english_spell = SpellChecker(language='en')
DetectorFactory.seed = 0
logging.info(f'✅ تم التحميل في {time.time()-start_time:.2f} ثانية')

# حفظ نماذج EasyOCR على Drive
drive_easyocr_path = '/content/drive/MyDrive/Handwriting_Dataset/.EasyOCR'
local_easyocr_path = os.path.expanduser('~/.EasyOCR')

if not os.path.exists(drive_easyocr_path) and os.path.exists(local_easyocr_path):
    print('جاري نقل نماذج EasyOCR إلى Drive للمرة الأولى...')
    !mv {local_easyocr_path} {drive_easyocr_path}

if not os.path.islink(local_easyocr_path):
    if os.path.exists(local_easyocr_path):
        !rm -rf {local_easyocr_path}
    !ln -s {drive_easyocr_path} {local_easyocr_path}
    print('✅ تم ربط نماذج EasyOCR بـ Google Drive بنجاح')


In [ ]:
# ============================================================
# الخلية 4: دوال المعالجة والتعرف والتصحيح
# ============================================================
import json
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, 'ocr_checkpoint.json')

def save_checkpoint(page_num, total_pages, processed_words):
    data = {
        'last_page_processed': page_num,
        'total_pages':         total_pages,
        'processed_words':     processed_words,
        'timestamp':           datetime.now().isoformat()
    }
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return None

def clear_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

def build_correction_dict():
    """يبني قاموس تصحيح تلقائي من تصحيحات المستخدم السابقة"""
    CORRECTION_DICT_PATH = os.path.join(OUTPUT_DIR, 'correction_dict.json')
    if not os.path.exists(FEEDBACK_CSV):
        return {}
    try:
        df_fb       = pd.read_csv(FEEDBACK_CSV, encoding='utf-8')
        corrections = {}
        for _, row in df_fb.iterrows():
            orig = str(row['original_text']).strip()
            corr = str(row['corrected_text']).strip()
            if orig and corr and orig != corr:
                if orig not in corrections:
                    corrections[orig] = {}
                corrections[orig][corr] = corrections[orig].get(corr, 0) + 1
        final_dict = {}
        for orig, candidates in corrections.items():
            best = max(candidates, key=candidates.get)
            if candidates[best] >= 2:
                final_dict[orig] = best
        with open(CORRECTION_DICT_PATH, 'w', encoding='utf-8') as f:
            json.dump(final_dict, f, ensure_ascii=False, indent=2)
        logging.info(f'✅ قاموس التصحيح: {len(final_dict)} كلمة')
        return final_dict
    except Exception as e:
        logging.error(f'خطأ في بناء القاموس: {e}')
        return {}

def apply_correction_dict(text, correction_dict):
    if not correction_dict or not text:
        return text
    return ' '.join([correction_dict.get(w, w) for w in text.split()])

def preprocess_image(img_bgr):
    gray      = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    clahe     = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced  = clahe.apply(gray)
    denoised  = cv2.fastNlMeansDenoising(enhanced, h=30)
    _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return binary

def smart_word_segmentation(img_bgr, binary_img, existing_detections=None):
    word_boxes = []
    if existing_detections:
        for (bbox, text, conf) in existing_detections:
            pts = np.array(bbox, dtype=np.int32)
            x, y, w, h = cv2.boundingRect(pts)
            word_boxes.append((x, y, w, h))
        return word_boxes
    kernel   = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 5))
    dilated  = cv2.dilate(binary_img, kernel, iterations=1)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        x, y, bw, bh = cv2.boundingRect(cnt)
        if bw > 15 and bh > 10:
            word_boxes.append((x, y, bw, bh))
    return word_boxes

def recognize_word_ensemble(img_bgr, easyocr_raw=None):
    results = []
    try:
        rgb    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        inputs = trocr_processor(images=Image.fromarray(rgb), return_tensors='pt').pixel_values.to(device)
        generated_ids = trocr_model.generate(inputs, max_length=50)
        trocr_text    = trocr_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        results.append(('trocr', trocr_text, 0.7))
    except:
        pass
    if easyocr_raw:
        results.append(('easyocr', easyocr_raw[1], easyocr_raw[2]))
    if not results:
        return '', 0.0, 'none', True
    best = max(results, key=lambda x: x[2])
    return best[1], best[2], best[0], (best[2] < 0.5)

def correct_text(text):
    if not text:
        return ''
    try:
        lang = detect(text)
        if lang == 'ar':
            return arabic_spell.contextual_correct(text)
        else:
            words     = text.split()
            corrected = [english_spell.correction(w) or w for w in words]
            return ' '.join(corrected)
    except:
        return text

print('✅ تم تعريف جميع دوال المعالجة')


In [ ]:
# ============================================================
# الخلية 5: دالة معالجة PDF
# ============================================================
def process_pdf(pages_start=1, pages_end=2, resume=True):
    proc_start = time.time()

    checkpoint = load_checkpoint() if resume else None
    if checkpoint:
        print(f"🔄 استئناف من الصفحة {checkpoint['last_page_processed']}")
        pages_start = checkpoint['last_page_processed']

    correction_dict = build_correction_dict()

    try:
        images = convert_from_path(
            PDF_PATH, dpi=300,
            first_page=pages_start, last_page=pages_end)
    except Exception as e:
        logging.error(f'فشل معالجة PDF: {e}')
        return

    total_words = checkpoint.get('processed_words', 0) if checkpoint else 0

    with sqlite3.connect(DB_PATH) as conn:
        conn.execute('''CREATE TABLE IF NOT EXISTS handwriting_data
            (image_id     INTEGER PRIMARY KEY AUTOINCREMENT,
             image_data   BLOB,
             predicted_text TEXT,
             status       TEXT,
             confidence   REAL,
             model_source TEXT,
             x INTEGER, y INTEGER, w INTEGER, h INTEGER,
             page_num     INTEGER)''')

        # ✅ حذف البيانات القديمة لتجنب التكرار
        conn.execute(
            'DELETE FROM handwriting_data WHERE page_num BETWEEN ? AND ?',
            (pages_start, pages_end))
        conn.commit()
        logging.info(f'🗑️ تم مسح بيانات الصفحات {pages_start}-{pages_end}')

        for idx, pil in enumerate(images):
            actual_page = pages_start + idx
            img_bgr     = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)

            try:
                detections = easy_reader.readtext(img_bgr, detail=1)
            except:
                detections = []

            binary     = preprocess_image(img_bgr)
            boxes_info = []

            if detections:
                for (bbox, text, conf) in detections:
                    pts = np.array(bbox, dtype=np.int32)
                    x, y, w, h = cv2.boundingRect(pts)
                    boxes_info.append(((x, y, w, h), (bbox, text, conf)))
            else:
                manual_boxes = smart_word_segmentation(img_bgr, binary)
                for box in manual_boxes:
                    boxes_info.append((box, None))

            for (box, e_raw) in boxes_info:
                x, y, w, h = box
                crop  = img_bgr[y:y+h, x:x+w]
                raw, conf, src, _ = recognize_word_ensemble(crop, easyocr_raw=e_raw)
                final = apply_correction_dict(correct_text(raw), correction_dict)
                _, buf = cv2.imencode('.png', crop)
                conn.execute(
                    '''INSERT INTO handwriting_data
                       (image_data, predicted_text, status, confidence,
                        model_source, x, y, w, h, page_num)
                       VALUES (?,?,?,?,?,?,?,?,?,?)''',
                    (buf.tobytes(), final, 'unverified', conf, src,
                     x, y, w, h, actual_page))
                total_words += 1

            save_checkpoint(actual_page + 1, pages_end, total_words)
            patches.cv2_imshow(cv2.resize(img_bgr, (0, 0), fx=0.3, fy=0.3))
        conn.commit()

    clear_checkpoint()
    stats = {
        'timestamp': datetime.now().isoformat(),
        'pages':     pages_end - pages_start + 1,
        'words':     total_words
    }
    with open(STATS_JSON, 'w', encoding='utf-8') as f:
        json.dump(stats, f, ensure_ascii=False)
    logging.info(f'✅ اكتملت المعالجة في {time.time()-proc_start:.2f} ثانية. كلمات: {total_words}')

print('✅ تم تعريف دالة process_pdf')


In [ ]:
# ============================================================
# الخلية 6: تشغيل الاستخراج — عدّل نطاق الصفحات حسب الحاجة
# ============================================================
process_pdf(pages_start=1, pages_end=2, resume=True)


## الخلية 7
يُرجى تشغيل الخلية 8 أولاً لتعريف الدوال، ثم استدعاء `launch_review_ui_v2()` من هناك.


In [ ]:
# ============================================================
# الخلية 8: دوال التصدير والتحليل والتدريب
# ============================================================
print('\n📁 ملفات المراقبة:')
print(f'   • سجل الأحداث:       {LOG_FILE}')
print(f'   • إحصائيات المعالجة: {STATS_JSON}')
print(f'   • تصحيحات المستخدم:  {FEEDBACK_CSV}')

# --- تصدير بيانات التدريب ---
def export_finetuning_dataset(output_dir='hf_training_dataset', val_ratio=0.1):
    import random
    os.makedirs(output_dir, exist_ok=True)
    img_dir = os.path.join(output_dir, 'images')
    os.makedirs(img_dir, exist_ok=True)

    with sqlite3.connect(DB_PATH) as conn:
        df_verified = pd.read_sql_query('''
            SELECT image_id, image_data, predicted_text, status
            FROM handwriting_data
            WHERE status IN ('verified', 'sentence_corrected')
            ORDER BY image_id
        ''', conn)

    if df_verified.empty:
        print('⚠️ لا توجد بيانات موثقة. قم بمراجعة بعض الكلمات وتأكيدها أولاً.')
        return None

    jsonl_records = []
    for _, row in df_verified.iterrows():
        filename = f"img_{row['image_id']}.png"
        with open(os.path.join(img_dir, filename), 'wb') as f:
            f.write(row['image_data'])
        jsonl_records.append({
            'image':    filename,
            'text':     row['predicted_text'].strip() if row['predicted_text'] else '',
            'verified': True
        })

    random.shuffle(jsonl_records)
    split_idx  = int(len(jsonl_records) * (1 - val_ratio))
    train_data = jsonl_records[:split_idx]
    val_data   = jsonl_records[split_idx:]

    def save_jsonl(data, fname):
        path = os.path.join(output_dir, fname)
        with open(path, 'w', encoding='utf-8') as f:
            for rec in data:
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')

    save_jsonl(train_data, 'train.jsonl')
    save_jsonl(val_data,   'val.jsonl')
    print(f'✅ تم التصدير: {len(jsonl_records)} عينة موثقة.')
    print(f'📁 المسار: {os.path.abspath(output_dir)}')
    return output_dir

# --- استخراج القاموس ثنائي اللغة ---
def extract_bilingual_vocab():
    """استخرج المفردات الإنجليزية-العربية من البيانات الموثقة"""
    with sqlite3.connect(DB_PATH) as conn:
        words = pd.read_sql_query("""
            SELECT DISTINCT predicted_text, page_num, x, y
            FROM handwriting_data
            WHERE status IN ('verified', 'sentence_corrected')
            ORDER BY page_num, y, x
        """, conn)

    if words.empty:
        print('⚠️ لا توجد كلمات موثقة.')
        return

    vocab_pairs = []
    for page in words['page_num'].unique():
        p_words   = words[words['page_num'] == page].copy()
        curr_line = [p_words.iloc[0].to_dict()]
        lines     = []
        for i in range(1, len(p_words)):
            row = p_words.iloc[i].to_dict()
            if abs(row['y'] - curr_line[-1]['y']) <= 30:
                curr_line.append(row)
            else:
                lines.append(curr_line)
                curr_line = [row]
        lines.append(curr_line)
        for line in lines:
            texts    = [str(w['predicted_text']) for w in line
                        if w['predicted_text'] and str(w['predicted_text']).strip()]
            en_words = [t for t in texts if t and all(ord(c) < 128 for c in t.replace(' ', ''))]
            ar_words = [t for t in texts if t and any('\u0600' <= c <= '\u06FF' for c in t)]
            if en_words or ar_words:
                vocab_pairs.append({
                    'english': ' | '.join(en_words),
                    'arabic':  ' | '.join(ar_words),
                    'page':    page
                })

    df     = pd.DataFrame(vocab_pairs)
    output = os.path.join(OUTPUT_DIR, 'bilingual_vocab.csv')
    df.to_csv(output, index=False, encoding='utf-8-sig')
    print(f'✅ تم استخراج {len(df)} زوج من المفردات')
    print(f'📁 المسار: {output}')
    display(df.head(20))
    return df

# --- إعادة بناء الجمل ---
def reconstruct_sentences(y_tolerance=25):
    with sqlite3.connect(DB_PATH) as conn:
        words = pd.read_sql_query(
            "SELECT * FROM handwriting_data WHERE status='verified' ORDER BY page_num, y, x",
            conn)
    if words.empty:
        return
    all_sentences = []
    for page in words['page_num'].unique():
        p_words   = words[words['page_num'] == page].copy()
        curr_line = [p_words.iloc[0].to_dict()]
        lines     = []
        for i in range(1, len(p_words)):
            row = p_words.iloc[i].to_dict()
            if abs(row['y'] - curr_line[-1]['y']) <= y_tolerance:
                curr_line.append(row)
            else:
                lines.append(curr_line)
                curr_line = [row]
        lines.append(curr_line)
        for line in lines:
            preview = ' '.join([str(w['predicted_text']) for w in line])
            try:
                lang = detect(preview)
            except:
                lang = 'en'
            sorted_line = sorted(line, key=lambda k: k['x'], reverse=(lang == 'ar'))
            sentence    = ' '.join([str(w['predicted_text']) for w in sorted_line])
            all_sentences.append({'page': page, 'text': sentence, 'lang': lang})
    df_res = pd.DataFrame(all_sentences)
    display(df_res)
    return df_res

# --- رفع إلى HuggingFace ---
from huggingface_hub import HfApi, login

def push_to_huggingface(hf_repo_id, hf_token, local_dataset_dir='hf_training_dataset'):
    if not os.path.exists(local_dataset_dir):
        return print('⚠️ شغّل export_finetuning_dataset() أولاً.')
    login(token=hf_token)
    api = HfApi()
    try:
        api.create_repo(repo_id=hf_repo_id, repo_type='dataset', exist_ok=True)
    except Exception as e:
        print(f'Repo info: {e}')
    api.upload_folder(
        folder_path=local_dataset_dir,
        repo_id=hf_repo_id,
        repo_type='dataset',
        commit_message=f"Update dataset - {datetime.now().strftime('%Y-%m-%d')}")
    print(f'✅ تم الرفع: https://huggingface.co/datasets/{hf_repo_id}')

# --- Fine-tuning بـ LoRA ---
from peft import get_peft_model, LoraConfig, TaskType
from torch.optim import AdamW
from torch.utils.data import Dataset as TorchDataset, DataLoader

def finetune_trocr_lora(min_samples=100):
    global trocr_model  # يجب أن يكون في البداية
    with sqlite3.connect(DB_PATH) as conn:
        df_v = pd.read_sql_query(
            "SELECT image_data, predicted_text FROM handwriting_data "
            "WHERE status IN ('verified', 'sentence_corrected')", conn)
    if len(df_v) < min_samples:
        return print(f'⚠️ لديك {len(df_v)} عينة فقط. انتظر حتى {min_samples}.')

    print(f'🚀 بدء التدريب على {len(df_v)} عينة...')
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM, r=16, lora_alpha=32,
        target_modules=['query', 'value'], lora_dropout=0.1)
    model = get_peft_model(trocr_model, lora_config)
    model.train()

    class HandwritingDataset(TorchDataset):
        def __init__(self, df):
            self.df = df
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            row    = self.df.iloc[idx]
            img    = Image.open(io.BytesIO(row['image_data'])).convert('RGB')
            pv     = trocr_processor(images=img, return_tensors='pt').pixel_values.squeeze()
            labels = trocr_processor.tokenizer(
                row['predicted_text'], return_tensors='pt',
                padding='max_length', max_length=50).input_ids.squeeze()
            labels[labels == trocr_processor.tokenizer.pad_token_id] = -100
            return {'pixel_values': pv, 'labels': labels}

    loader    = DataLoader(HandwritingDataset(df_v), batch_size=4, shuffle=True)
    optimizer = AdamW(model.parameters(), lr=5e-5)

    for epoch in range(3):
        total_loss = 0
        for batch in loader:
            out = model(
                pixel_values=batch['pixel_values'].to(device),
                labels=batch['labels'].to(device))
            out.loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += out.loss.item()
        print(f"Epoch {epoch+1}/3 | Loss: {total_loss/len(loader):.4f}")

    lora_path = os.path.join(MODEL_CACHE, 'trocr_lora_finetuned')
    model.save_pretrained(lora_path)
    trocr_processor.save_pretrained(lora_path)
    trocr_model = model
    print(f'✅ تم الحفظ في: {lora_path}')

# --- تعديل القاموس يدوياً ---
def edit_correction_dict_ui():
    from IPython.display import clear_output
    dict_path = os.path.join(OUTPUT_DIR, 'correction_dict.json')
    data = {}
    if os.path.exists(dict_path):
        with open(dict_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    out    = widgets.Output()
    key_in = widgets.Text(description='أصلي:')
    val_in = widgets.Text(description='تصحيح:')
    def save(b):
        if key_in.value:
            data[key_in.value] = val_in.value
            with open(dict_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False)
            with out:
                clear_output()
                print(f'تم حفظ: {key_in.value} → {val_in.value}')
    btn = widgets.Button(description='حفظ', button_style='success')
    btn.on_click(save)
    display(widgets.VBox([key_in, val_in, btn, out]))

# edit_correction_dict_ui()  # ألغِ التعليق للتشغيل

# --- واجهة مراجعة الكلمات v2 ---
def launch_review_ui_v2():
    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query(
            "SELECT * FROM handwriting_data WHERE status='unverified' ORDER BY confidence ASC",
            conn)
    if df.empty:
        print('✅ لا توجد كلمات للمراجعة.')
        return

    current    = 0
    img        = widgets.Image(format='png', width=350)
    text       = widgets.Text(description='النص:')
    conf_bar   = widgets.FloatProgress(min=0, max=1.0, description='الثقة:', layout={'width': '95%'})
    conf_label = widgets.HTML(value='')
    prog       = widgets.IntProgress(min=0, max=len(df)-1, bar_style='info')
    info       = widgets.Label()

    def update():
        nonlocal current
        if 0 <= current < len(df):
            row = df.iloc[current]
            img.value        = row['image_data']
            text.value       = str(row['predicted_text'] or '')
            c                = float(row['confidence'])
            conf_bar.value   = c
            conf_bar.bar_style = 'danger' if c < 0.5 else ('warning' if c < 0.8 else 'success')
            conf_label.value = f'<b>{c:.2%}</b>'
            info.value       = f"{current+1}/{len(df)} | صفحة {row['page_num']}"
            prog.value       = current
        else:
            img.value  = b''
            text.value = ''
            info.value = '🏁 اكتملت المراجعة'

    def on_confirm(b):
        nonlocal current, df
        if not (0 <= current < len(df)):
            return
        rid       = int(df.iloc[current]['image_id'])
        original  = df.iloc[current]['predicted_text']
        corrected = text.value
        with sqlite3.connect(DB_PATH) as conn:
            conn.execute(
                "UPDATE handwriting_data SET predicted_text=?, status='verified' WHERE image_id=?",
                (corrected, rid))
        if original != corrected:
            pd.DataFrame([{
                'timestamp':      datetime.now().isoformat(),
                'image_id':       rid,
                'original_text':  original,
                'corrected_text': corrected,
                'status':         'verified'
            }]).to_csv(FEEDBACK_CSV, mode='a',
                       header=not os.path.exists(FEEDBACK_CSV),
                       index=False, encoding='utf-8')
        df = df.drop(df.index[current]).reset_index(drop=True)
        prog.max = max(0, len(df)-1)
        if current >= len(df) and len(df) > 0:
            current = len(df) - 1
        update()

    def on_next(b):
        nonlocal current
        current = min(len(df)-1, current + 1)
        update()

    def on_prev(b):
        nonlocal current
        current = max(0, current - 1)
        update()

    def on_delete(b):
        nonlocal current, df
        if not (0 <= current < len(df)):
            return
        rid = int(df.iloc[current]['image_id'])
        with sqlite3.connect(DB_PATH) as conn:
            conn.execute('DELETE FROM handwriting_data WHERE image_id=?', (rid,))
        df = df.drop(df.index[current]).reset_index(drop=True)
        prog.max = max(0, len(df)-1)
        if current >= len(df) and len(df) > 0:
            current = len(df) - 1
        update()

    btn_confirm = widgets.Button(description='✅ تأكيد',    button_style='success')
    btn_next    = widgets.Button(description='التالي',      button_style='info')
    btn_prev    = widgets.Button(description='⬅️ السابق',   button_style='info')
    btn_del     = widgets.Button(description='🗑️ حذف',     button_style='danger')

    btn_confirm.on_click(on_confirm)
    btn_next.on_click(on_next)
    btn_prev.on_click(on_prev)
    btn_del.on_click(on_delete)

    display(widgets.VBox([
        prog, info, img, text,
        widgets.HBox([conf_bar, conf_label]),
        widgets.HBox([btn_prev, btn_confirm, btn_del, btn_next])
    ]))
    update()

# ✅ الاستدعاء بعد التعريف
launch_review_ui_v2()


In [ ]:
# ============================================================
# الخلية 9: واجهة مراجعة الجمل
# ============================================================
def launch_sentence_review_ui(y_tolerance=25):
    import io as _io

    with sqlite3.connect(DB_PATH) as conn:
        words_df = pd.read_sql_query(
            'SELECT * FROM handwriting_data ORDER BY page_num, y, x', conn)

    if words_df.empty:
        return print('⚠️ لا توجد بيانات للمراجعة.')

    sentences = []
    for page in words_df['page_num'].unique():
        p_words   = words_df[words_df['page_num'] == page].copy()
        if p_words.empty:
            continue
        curr_line = [p_words.iloc[0].to_dict()]
        for i in range(1, len(p_words)):
            row = p_words.iloc[i].to_dict()
            if abs(row['y'] - curr_line[-1]['y']) <= y_tolerance:
                curr_line.append(row)
            else:
                sentences.append(curr_line)
                curr_line = [row]
        sentences.append(curr_line)

    if not sentences:
        return print('⚠️ لم يتم تكوين أي جمل.')

    current_idx    = 0
    img_container  = widgets.HBox(layout={'overflow_x': 'scroll', 'padding': '10px'})
    sentence_input = widgets.Textarea(
        description='الجملة:',
        layout={'width': '95%', 'height': '80px'})
    info_label = widgets.Label()
    progress   = widgets.IntProgress(min=0, max=len(sentences)-1, layout={'width': '95%'})

    def get_img_widget(blob):
        img = Image.open(_io.BytesIO(blob))
        buf = _io.BytesIO()
        img.save(buf, format='PNG')
        return widgets.Image(value=buf.getvalue(), format='png', width=120)

    def update_ui():
        nonlocal current_idx
        if not (0 <= current_idx < len(sentences)):
            return
        sent = sentences[current_idx]
        img_container.children = [get_img_widget(w['image_data']) for w in sent]
        original_text          = ' '.join([str(w['predicted_text'] or '') for w in sent])
        sentence_input.value   = original_text
        info_label.value       = (
            f"جملة {current_idx+1} من {len(sentences)} "
            f"| صفحة {sent[0]['page_num']}")
        progress.value = current_idx

    def save_current(b):
        nonlocal current_idx
        sent      = sentences[current_idx]
        original  = ' '.join([str(w['predicted_text'] or '') for w in sent])
        corrected = sentence_input.value.strip()
        if not corrected:
            return
        with sqlite3.connect(DB_PATH) as conn:
            for w in sent:
                conn.execute(
                    "UPDATE handwriting_data SET status='sentence_corrected' WHERE image_id=?",
                    (w['image_id'],))
        sent_id = f"p{sent[0]['page_num']}_y{sent[0]['y']}"
        if original != corrected:
            pd.DataFrame([{
                'timestamp':      datetime.now().isoformat(),
                'image_id':       None,
                'original_text':  original,
                'corrected_text': corrected,
                'status':         f'sent_rev_{sent_id}'
            }]).to_csv(FEEDBACK_CSV, mode='a',
                       header=not os.path.exists(FEEDBACK_CSV),
                       index=False, encoding='utf-8')
            orig_words = original.split()
            corr_words = corrected.split()
            if len(orig_words) == len(corr_words):
                derived = [
                    {'timestamp':      datetime.now().isoformat(),
                     'image_id':       None,
                     'original_text':  o,
                     'corrected_text': c,
                     'status':         'sentence_derived'}
                    for o, c in zip(orig_words, corr_words) if o != c
                ]
                if derived:
                    pd.DataFrame(derived).to_csv(
                        FEEDBACK_CSV, mode='a', header=False,
                        index=False, encoding='utf-8')
        print(f'✅ تم حفظ الجملة {current_idx+1}')
        current_idx = min(len(sentences)-1, current_idx + 1)
        update_ui()

    def on_next(b):
        nonlocal current_idx
        current_idx = min(len(sentences)-1, current_idx + 1)
        update_ui()

    def on_prev(b):
        nonlocal current_idx
        current_idx = max(0, current_idx - 1)
        update_ui()

    btn_save = widgets.Button(description='✅ حفظ وتأكيد', button_style='success')
    btn_next = widgets.Button(description='التالي ➡',      button_style='info')
    btn_prev = widgets.Button(description='⬅ السابق',      button_style='info')

    btn_save.on_click(save_current)
    btn_next.on_click(on_next)
    btn_prev.on_click(on_prev)

    display(widgets.VBox([
        progress, info_label, img_container,
        sentence_input,
        widgets.HBox([btn_prev, btn_save, btn_next])
    ]))
    update_ui()

launch_sentence_review_ui()


## دليل الاستخدام

```
ترتيب التشغيل في كل جلسة:
════════════════════════════
1. شغّل الخلايا 1-4 (تثبيت + إعداد + نماذج + دوال)
2. process_pdf(pages_start=X, pages_end=Y)   ← استخراج
3. launch_review_ui_v2()                     ← مراجعة كلمات
4. launch_sentence_review_ui()               ← مراجعة جمل
5. extract_bilingual_vocab()                 ← قاموس ثنائي اللغة
6. export_finetuning_dataset()               ← تصدير بيانات التدريب
7. push_to_huggingface("DrAbdulmalek/arabic-medical-handwriting", HF_TOKEN)
8. finetune_trocr_lora()                     ← بعد 100+ عينة موثقة

الجلسة التالية ستستفيد تلقائياً من:
  - قاموس التصحيح المبني من تصحيحاتك
  - نموذج LoRA المحسّن (إذا دُرّب)
```
